In [6]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = True
_AUGMENTED_PARAM = 'x_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.05
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = True
_MC_INCLUDE_BY_PREDICTOR = False
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)



In [7]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83  -0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [8]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [9]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [10]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [11]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: True
Augmented measurement equation: OutGap
Augmented coefficient: x_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[0.604 0.    0.   ]
 [0.    0.839 0.   ]
 [0.    0.    0.039]]


In [12]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.470,0.422,0.006,0.001,100000,10186,0.102,0.001,0.100,0.104
1,Infl,0.860,0.523,0.004,0.001,100000,3432,0.034,0.001,0.033,0.035
2,Rate,0.988,0.501,0.004,0.001,100000,4793,0.048,0.001,0.047,0.049


In [13]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 100000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.120,3.084,0.508,0.000,0.009,0.001,100000,6646,0.066,0.001,0.065,0.068,3.0,200,4
1,cov_identity,4.186,731.053,0.000,0.002,0.843,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


In [14]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-0.236,-0.013,1.336,-0.188,0.495,0.005,0.004,0.0,0.000,0.003,0.001,0.0,100000,5394,0.054,0.001,0.053,0.055
1,OutGap,x,-0.323,-0.088,0.248,-1.259,0.306,0.012,0.001,0.0,0.000,0.003,0.001,0.0,100000,22947,0.229,0.001,0.227,0.232
2,OutGap,r,-1.594,-0.052,2.125,-0.733,0.426,0.007,0.007,0.0,0.001,0.003,0.001,0.0,100000,10748,0.107,0.001,0.106,0.109
3,Infl,Pi,-0.545,-0.023,1.593,-0.330,0.481,0.006,0.005,0.0,0.000,0.003,0.001,0.0,100000,6433,0.064,0.001,0.063,0.066
4,Infl,x,0.019,0.006,0.297,0.083,0.493,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5473,0.055,0.001,0.053,0.056
5,Infl,r,-0.373,-0.008,2.539,-0.117,0.493,0.005,0.008,0.0,0.001,0.003,0.001,0.0,100000,5481,0.055,0.001,0.053,0.056
6,Rate,Pi,-0.067,-0.016,0.302,-0.226,0.487,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5917,0.059,0.001,0.058,0.061
7,Rate,x,0.025,0.030,0.056,0.428,0.468,0.006,0.000,0.0,0.000,0.003,0.001,0.0,100000,7494,0.075,0.001,0.073,0.077
8,Rate,r,-0.034,-0.003,0.482,-0.044,0.494,0.005,0.002,0.0,0.000,0.003,0.001,0.0,100000,5581,0.056,0.001,0.054,0.057


In [15]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-1.382,-0.118,0.818,-1.679,0.191,0.017,0.002,0.0,0.000,0.003,0.001,0.0,100000,36158,0.362,0.002,0.359,0.365
1,OutGap,x,-0.310,-0.141,0.150,-2.015,0.117,0.023,0.000,0.0,0.000,0.003,0.001,0.0,100000,51675,0.517,0.002,0.514,0.520
0,OutGap,r,-0.457,-0.016,2.039,-0.228,0.552,0.004,0.005,0.0,0.001,0.003,0.001,0.0,100000,2162,0.022,0.000,0.021,0.023
5,Infl,Pi,-0.303,-0.021,0.982,-0.291,0.487,0.005,0.003,0.0,0.000,0.003,0.001,0.0,100000,5991,0.060,0.001,0.058,0.061
4,Infl,x,-0.037,-0.011,0.181,-0.158,0.497,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5193,0.052,0.001,0.051,0.053
3,Infl,r,-0.054,-0.001,2.431,-0.011,0.503,0.005,0.008,0.0,0.001,0.003,0.001,0.0,100000,4789,0.048,0.001,0.047,0.049
8,Rate,Pi,0.021,0.007,0.186,0.099,0.501,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,4895,0.049,0.001,0.048,0.050
7,Rate,x,0.013,0.023,0.034,0.325,0.486,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,6101,0.061,0.001,0.060,0.063
6,Rate,r,-0.059,-0.006,0.461,-0.089,0.501,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5058,0.051,0.001,0.049,0.052


In [25]:
print("Innovation decomposition orthogonal summary:")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"])

Innovation decomposition orthogonal summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.775253,-2.010909,-0.235656,-0.235656,7.485784e-19,3.548687e-16,1.764399e-15,0.002692,0.002154,0.004234,0.004234,1.449148e-18,9.168988e-19,8.418804e-19
1,OutGap,x,-0.000675,-0.322404,-0.323079,-0.323079,-2.098143e-19,6.242446e-17,1.764399e-15,0.000507,0.000449,0.000815,0.000815,2.704717e-19,1.848969e-19,8.418804e-19
2,OutGap,r,-0.176316,-1.417243,-1.593560,-1.593560,-1.028781e-18,4.747604e-16,1.764399e-15,0.004362,0.003671,0.006798,0.006798,2.020856e-18,1.352728e-18,8.418804e-19
3,Infl,Pi,-0.036466,-0.508499,-0.544966,-0.544966,-1.750482e-17,3.863033e-16,1.764399e-15,0.001150,0.005011,0.005124,0.005124,1.605320e-18,1.042976e-18,8.418804e-19
4,Infl,x,0.005665,0.013252,0.018917,0.018917,-3.164757e-19,7.124726e-17,1.764399e-15,0.000215,0.000946,0.000964,0.000964,2.961535e-19,1.922125e-19,8.418804e-19
5,Infl,r,-0.036465,-0.336713,-0.373178,-0.373178,-1.282956e-18,6.137352e-16,1.764399e-15,0.001847,0.008138,0.008290,0.008290,2.561176e-18,1.671195e-18,8.418804e-19
6,Rate,Pi,-0.000421,-0.066473,-0.066895,-0.066895,6.882535e-20,1.403510e-16,1.764399e-15,0.000248,0.000958,0.000974,0.000974,5.617602e-19,3.443671e-19,8.418804e-19
7,Rate,x,-0.000069,0.025377,0.025308,0.025308,1.441708e-19,2.620272e-17,1.764399e-15,0.000046,0.000185,0.000185,0.000185,1.054551e-19,6.523102e-20,8.418804e-19
8,Rate,r,-0.010059,-0.024195,-0.034254,-0.034254,2.118216e-19,2.298859e-16,1.764399e-15,0.000395,0.001584,0.001578,0.001578,9.247088e-19,5.714947e-19,8.418804e-19


In [26]:
print("Innovation decomposition raw summary:")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"])

Innovation decomposition raw summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.850144,-3.232550,-1.382406,-1.382406,-2.011286e-18,3.484695e-16,1.764399e-15,0.001639,0.000920,0.002257,0.002257,1.422724e-18,8.999233e-19,8.418804e-19
1,OutGap,x,0.267348,-0.577238,-0.309890,-0.309890,1.053736e-19,6.569281e-17,1.764399e-15,0.000295,0.000281,0.000447,0.000447,2.725273e-19,1.763954e-19,8.418804e-19
2,OutGap,r,-1.044902,0.588269,-0.456633,-0.456633,3.358536e-18,4.396244e-16,1.764399e-15,0.005172,0.003245,0.005345,0.005345,1.826095e-18,1.184070e-18,8.418804e-19
3,Infl,Pi,-0.004462,-0.298393,-0.302856,-0.302856,-1.935371e-17,2.385835e-16,1.764399e-15,0.000710,0.003048,0.003118,0.003118,9.855522e-19,6.370495e-19,8.418804e-19
4,Infl,x,0.000528,-0.037057,-0.036529,-0.036529,-2.462310e-18,4.398196e-17,1.764399e-15,0.000131,0.000569,0.000579,0.000579,1.823825e-19,1.182357e-19,8.418804e-19
5,Infl,r,-0.023222,-0.030501,-0.053724,-0.053724,1.025320e-17,5.810209e-16,1.764399e-15,0.001760,0.007546,0.007671,0.007671,2.415651e-18,1.568608e-18,8.418804e-19
6,Rate,Pi,0.000034,0.021197,0.021231,0.021231,1.885917e-19,8.631733e-17,1.764399e-15,0.000153,0.000571,0.000586,0.000586,3.454820e-19,2.117796e-19,8.418804e-19
7,Rate,x,0.000019,0.012538,0.012557,0.012557,5.618229e-20,1.595407e-17,1.764399e-15,0.000028,0.000109,0.000110,0.000110,6.409275e-20,3.952928e-20,8.418804e-19
8,Rate,r,-0.006712,-0.052770,-0.059482,-0.059482,1.347709e-18,2.207539e-16,1.764399e-15,0.000378,0.001476,0.001473,0.001473,8.853229e-19,5.445061e-19,8.418804e-19


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.

In [16]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()



## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [17]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.315,-1335.19,-1049.074,572.232,0.0,0.0,0.17,0.05,0.268,0.0,100000,100000,1.0,0.0,1.0,1.0


In [18]:
res_mle

OptimizationResult(kind='mle', x=array([1.18874398]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(1.1887439820910692), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', fun=np.float64(1026.9796329523338), loglik=np.float64(-1026.9796329523338), logprior=np.float64(0.0), logpost=np.float64(-1026.9796329523338), nfev=14, nit=6, raw=  message: CONVERGENCE: NO

## Serial Autocorrelation Tests for the Augmented Model

In [19]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.860,0.373,0.007,0.001,100000,15269,0.153,0.001,0.150,0.155
1,Infl,4.335,0.168,0.012,0.001,100000,45522,0.455,0.002,0.452,0.458
2,Rate,1.001,0.498,0.004,0.001,100000,4928,0.049,0.001,0.048,0.051


In [20]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.120,3.084,0.508,0.000,0.009,0.001,100000,6646,0.066,0.001,0.065,0.068,3.0,200,4
1,cov_identity,4.186,731.053,0.000,0.002,0.843,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.096,2.941,0.514,0.0,0.008,0.001,100000,5262,0.053,0.001,0.051,0.054,3.0,200,4
1,cov_identity,0.562,159.742,0.000,0.0,0.234,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


""
